# Instructions
Before you start this lesson please click the "Run all" button at the top of the page:

![Screenshot showing run all button](https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/run-all-button.png?raw=true)

This might take about a minute or more.

In [ ]:
from io import BytesIO

import ipywidgets as widgets
from IPython.display import display, HTML
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
import cv2

# Images and Convolutions
In this lesson we're going to learn how we can take images and use them in neural networks.

Computers only know how to do things with numbers. In our previous data modeling activity we took concepts related to weather and represented them as numbers:

- Humidity: A percentage from 0% to 100%
- Temperature: A number in Farenheight
- Cloud Coverage: A percentage of how much of the sky is obscured by clouds from 0% to 100%

These weather measurements are easily translated into numbers. But how do we take something like images and turn those into numbers?

## Pixels
To learn how computers "see" images as numbers we need to learn about pixels.

To a computer an image is a series of squares which are each one color. We call these squares pixels.

You're probably already familiar with pixels even if you haven't heard the term. When a photo or video is really low quality can see each individual pixel.

Like in this photo:

<img alt="Low resolution photo of a cat" src="https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/low-resolution-photo-screenshot.png?raw=true" width="400px" />

Look at the ears and face of the cat. You can clearly see boxes which are each one color (pixels).

You can think of an image then as a list of pixels.

## Colors
So if an image is made up of a bunch of pixels. And each pixel is an individual color. Then how do we represent colors as numbers?

Colors can be represented by mixing the primary colors red, green, and blue together in different amounts. If you've taken an art class then this may be familiar to you.

So to represent colors in computers we just keep track of how much red, blue, and green are in the color.

Play with the demo below to see how you can make colors out of combinations of red, green, and blue:

In [ ]:
# @title
def update_color(r, g, b):
    color_hex = f'#{r:02x}{g:02x}{b:02x}'
    display(HTML(f"<div style='width:100px; height:100px; background-color:{color_hex}; border: 1px solid black;'></div>"))

red_slider = widgets.IntSlider(min=0, max=255, step=1, value=113, description='Red:')
green_slider = widgets.IntSlider(min=0, max=255, step=1, value=60, description='Green:')
blue_slider = widgets.IntSlider(min=0, max=255, step=1, value=255, description='Blue:')


output = widgets.interactive_output(update_color, {'r': red_slider, 'g': green_slider, 'b': blue_slider})

display(red_slider, green_slider, blue_slider, output)

In computers we track how much of each primary color is present using a number between 0 and 255 (0 meaning none of the color, 255 meaning all of the color).

> **Why 0 to 255?**  
> This number is convenient for computers to store. As 255 is the biggest number that can be represented with 8 ones and zeros.

In order to store this data we use a list. Where the:

- First item is the amount of red
- Second item is the amount of blue
- Third item is the amount of green

So for example:

In [ ]:
# [red, green, blue]
color = [113, 60, 255]

This list above represents a blue-ish purple color.

## Images as Pixels with Colors
So far we've learned that:

- An image is just a list of pixels
- Each pixel is a color
- Each color is just a list of the amount of red, green, and blue

Then an image is really a list of colors. For example this could represent an image:

In [ ]:
image = [
    [0, 34, 87],
    [90, 46, 1],
    [0, 0, 0],
    [89, 30, 111],
]

In [ ]:
# @title
# Convert the list of lists (RGB pixels) to a NumPy array of type uint8
image_array = np.array(image, dtype=np.uint8)

# Reshape the array to (height, width, channels) for a horizontal strip of pixels
# Here, height is 1, width is the number of pixels (len(image)), and channels is 3 (RGB)
display_image_array = image_array.reshape(1, len(image), 3)

# Display the image using matplotlib
plt.imshow(display_image_array)
plt.axis('off') # Hide axes for a cleaner image display
plt.show()

# Check In - Images
We just learned how images are converted into numbers. Here are the things you should know:

- Images are made of up a list of pixels
- Each pixel is square which is one color
- Colors are made by mixing the primary colors: red, green, and blue
- Colors are represented as numbers by the amount of red, green, and blue in the color
- Use numbers between 0 and 255

# Understanding Images as a Computer
So now we know how images are represented as numbers. But how do computers "look" at those images and understand what is in an image?

- A first approach might be to just let the computer read all the numbers for each pixel
- But that's a ton of data
- And doesn't really match how we as humans understand images...

## How do Humans Understand Images?
Before we learn how computers understand images, let's think about how humans understand images.

When you look at a photo what do you pick out?

- Corners?
- Lines?
- Objects?
- Contrast between light and dark areas?

When humans look use their eyes we aren't looking at each individual pixel on its own. We're looking at patterns in the image. We're picking out features of objects.

## Transfering this to Computers
Okay, so if humans don't look at individual pixels, but instead pick up on patterns in the image. Then how can computers do this?

For computers to pick up on patterns in images we use something called "convolutions". This might sound like a crazy convoluted word. But, you probably have used convolutions in your daily life before...

# Convolutions
Convolutions are the same technology behind photo filters like those in Instagram, snapchat, and others. As well image editing tools like Photoshop.

For example, here is that cat photo from earlier but run through a bunch of different types of convolutions:


In [ ]:
# @title
response = requests.get("https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/medium-resolution-photo.jpg?raw=true")
img = np.array(Image.open(BytesIO(response.content)))

def blur(src):
  kernel = np.array([
      [1/9, 1/9, 1/9],
      [1/9, 1/9, 1/9],
      [1/9, 1/9, 1/9],
  ])

  return cv2.filter2D(src, -1, kernel)

def sharpen(src):
  kernel = np.array([
      [0, -1, 0],
      [-1, 5, -1],
      [0, -1, 0]
  ])

  return cv2.filter2D(src, -1, kernel)

def emboss(src):
  kernel = np.array([
      [-2, -1, 0],
      [-1, 1, 1],
      [0, 1, 2]
  ])

  return cv2.filter2D(src, -1, kernel)


def sepia(src):
    kernel = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131]
    ])
    after = cv2.transform(src, kernel)
    return np.clip(after, 0, 255).astype(np.uint8)

def blue(src):
    kernel = np.array([
         [0.25, 0.4, 0.15],
         [0.35, 0.65, 0.25],
         [0.4, 0.75, 0.35],
    ])
    after = cv2.transform(src, kernel)
    return np.clip(after, 0, 255).astype(np.uint8)

def left_sobel(src):
    kernel = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ])
    return cv2.filter2D(src, -1, kernel)

def outline(src):
    kernel = np.array([
        [-1, -1, -1],
        [-1, 8, -1],
        [-1, -1, -1],
    ])
    return cv2.filter2D(src, -1, kernel)

show_images(img, blur(img), "Blurred")
show_images(img, sharpen(img), "Sharpened")
show_images(img, emboss(img), "Embossed")
show_images(img, sepia(img), "Sepia")
show_images(img, blue(img), "Blue")
show_images(img, left_sobel(img), "Left Sobel")
show_images(img, outline(img), "Outline")

So a convolution changes how an image looks. But what exactly is a convolution?

## Step by Step
A convolution changes each pixel in our image by a little bit.

To do this we:

1. Arrange the pixels in our image in an X, Y grid like so:
  ![Screenshot of cat super zoomed in](https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/low-resolution-photo-super-zoomed-screenshot.png?raw=true)
2. For each pixel we look at its direct neighbors:  
  _(Target pixel we are trying to change highlighted in blue)_  
  _(Surrounding neighbor pixels highlighted in red)_  
  ![Target pixel highlighted in blue, neighbor pixels highlighted in red](https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/low-resolution-photo-super-zoomed-screenshot-kernel-area.png?raw=true)
3. Get the new value of the pixel by:
  - Multiply each neighbor pixel's color by a number we define
    - Remember: A pixel's color is defined by 3 numbers
    - The amount of red, green, and blue
    - Ranging from 0 to 255
    - So we just multiply each of these numbers to get the new red, green, and blue color
  - Add up the multiplied values together
  - Use this new value as the center pixel's new color

We complete this process for every pixel in the image. Getting its new color by multiplying and adding up the color values of their neighboring pixels.

## Kernels
So how are we able to get so many different effects just by completing this same process? The answer lies in which number we multiply the color values by.

- We can multiply each neighboring pixel by a different value based on its position
- We can multiply neighboring pixels by different amounts
- We can also multiply the target pixel by a number and include that

These numbers that we choose are sort of the like the paint brush type our convolution uses to paint the new pixel colors. We call this paint brush a **kernel**.

Let's look at an example kernel to get a better sense of them:

![Image of sobel kernel numbers](https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/low-resolution-photo-super-zoomed-screenshot-kernel-numbers.png?raw=true)

To get the new color of the pixel in blue we multiply each pixel and add them up:

- First / top row
  - Left by -1
  - Center by 0
  - Right by 1
- Second / middle row
  - Left by -2
  - Center by 0
  - Right by 2
- Third / last row
  - Left by -1
  - Center by 0
  - Right by 1

That's a lot to write out. So what we do is record our kernel in a list. Where each item in the list indicates what number we multiply that row of pixels by. The same kernel as above written as a list is:

```python
kernel = [
  [-1, 0, 1],
  [-2, 0, 2],
  [-1, 0, 1]
]
```

The kernels for each of the effects in the photos above are:

In [ ]:
blur = [
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9],
]

sharpen = [
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
]

emboss = [
    [-2, -1, 0],
    [-1, 1, 1],
    [0, 1, 2]
]

sepia = [
    [0.393, 0.769, 0.189],
    [0.349, 0.686, 0.168],
    [0.272, 0.534, 0.131]
]

blue = [
    [0.25, 0.4, 0.15],
    [0.35, 0.65, 0.25],
    [0.4, 0.75, 0.35],
]

left_sobel = [
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
]

outline = [
    [-1, -1, -1],
    [-1, 8, -1],
    [-1, -1, -1],
]

# Check In - Convolutions
We just learned that convolutions are ways to apply effects to an image. Here are the things you should know:

- A convolution sets new color values for each pixel in an image
- This is done by looking at the neighboring pixels of each pixel
- Each neighboring pixel is multiplied by a value from a kernel, then added up, and used as the new pixel color
- The kernel influences what effect the convolution has on the image

# Use of Convolutions
So we can make some pretty funky looking photos with convolutions. But how does that help a computer understand what is in an image?

Well we can pick specific kernels which make the resulting image show some useful things:

- Edges of objects
- Overall shape of objects
- Centers of objects

For example:

In [ ]:
# @title
response = requests.get("https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/medium-resolution-photo.jpg?raw=true")
img = np.array(Image.open(BytesIO(response.content)))

def blur(src):
  kernel = np.array([
      [1/9, 1/9, 1/9],
      [1/9, 1/9, 1/9],
      [1/9, 1/9, 1/9],
  ])

  return cv2.filter2D(src, -1, kernel)

def sharpen(src):
  kernel = np.array([
      [0, -1, 0],
      [-1, 5, -1],
      [0, -1, 0]
  ])

  return cv2.filter2D(src, -1, kernel)

def emboss(src):
  kernel = np.array([
      [-2, -1, 0],
      [-1, 1, 1],
      [0, 1, 2]
  ])

  return cv2.filter2D(src, -1, kernel)


def sepia(src):
    kernel = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131]
    ])
    after = cv2.transform(src, kernel)
    return np.clip(after, 0, 255).astype(np.uint8)

def blue(src):
    kernel = np.array([
         [0.25, 0.4, 0.15],
         [0.35, 0.65, 0.25],
         [0.4, 0.75, 0.35],
    ])
    after = cv2.transform(src, kernel)
    return np.clip(after, 0, 255).astype(np.uint8)

def left_sobel(src):
    kernel = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ])
    return cv2.filter2D(src, -1, kernel)

def outline(src):
    kernel = np.array([
        [-1, -1, -1],
        [-1, 8, -1],
        [-1, -1, -1],
    ])
    return cv2.filter2D(src, -1, kernel)

def show_images(before, after, name):
    plt.figure(figsize=(14, 6), dpi=100)
    plt.subplot(1, 2, 1)
    plt.title("Original")
    plt.imshow(np.array(before))
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(name)
    plt.imshow(np.array(after))
    plt.axis('off')

    plt.show()

show_images(img, blur(blur(blur(blur(blur(blur(blur(blur(blur(img))))))))), "Blurred")
show_images(img, left_sobel(img), "Edges (Sobel)")
show_images(img, outline(img), "Edges (Outline)")